# Task 4 — Join, Transformation Rules and Testing -Hanin

## Saudi FinHub — Economic Indicators

This notebook prepares the validated economic indicator datasets for analysis by:

- Converting monthly inflation data to quarterly data
- Calculating nominal GDP growth
- Joining GDP, inflation, and unemployment using a common quarterly key
- Testing the transformation rules and final output

## Load Validated Economic Datasets - Hanin

Load the validated GDP, Inflation, and Unemployment datasets produced in Task 3.

In [6]:
import pandas as pd
from pathlib import Path

# Define data folders
INTERIM_DATA_PATH = Path("../data/interim")
PROCESSED_DATA_PATH = Path("../data/processed")

PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)

# Load validated datasets
gdp = pd.read_csv(
    INTERIM_DATA_PATH / "gdp_validated.csv"
)

inflation = pd.read_csv(
    INTERIM_DATA_PATH / "inflation_validated.csv",
    parse_dates=["date"],
    dayfirst=True
)

unemployment = pd.read_csv(
    INTERIM_DATA_PATH / "unemployment_validated.csv"
)

# Check the loaded datasets
print("GDP shape:", gdp.shape)
print("Inflation shape:", inflation.shape)
print("Unemployment shape:", unemployment.shape)

GDP shape: (53, 5)
Inflation shape: (151, 6)
Unemployment shape: (53, 6)


## Transformation 1 — Monthly Inflation to Quarterly Average - Hanin

Convert the monthly inflation rate into a quarterly average so it can be aligned with the quarterly GDP and Unemployment datasets.

In [7]:
# Calculate quarterly average inflation

inflation_quarterly = (
    inflation
    .groupby(
        ["year", "quarter", "year_quarter"],
        as_index=False
    )["inflation_rate"]
    .mean()
)

# Rename the aggregated column
inflation_quarterly = inflation_quarterly.rename(
    columns={
        "inflation_rate": "quarterly_inflation_rate"
    }
)

# Round for readability
inflation_quarterly["quarterly_inflation_rate"] = (
    inflation_quarterly["quarterly_inflation_rate"].round(2)
)

inflation_quarterly.head()

,year,quarter,year_quarter,quarterly_inflation_rate
0,2014,Q1,2014-Q1,1.37
1,2014,Q2,2014-Q2,2.03
2,2014,Q3,2014-Q3,2.00
3,2014,Q4,2014-Q4,1.97
4,2015,Q1,2015-Q1,1.53


## Transformation 2 — Nominal GDP Growth - Hanin

Calculate the year-over-year nominal GDP growth rate by comparing each quarter with the same quarter in the previous year.

In [10]:
# Sort GDP data chronologically
gdp = gdp.sort_values(
    by=["year", "quarter"]
).reset_index(drop=True)

# Calculate year-over-year nominal GDP growth
gdp["nominal_gdp_growth_rate"] = (
    gdp["gdp_value"].pct_change(periods=4) * 100
).round(2)

gdp.head(8)

,main_activities,year,quarter,year_quarter,gdp_value,nominal_gdp_growth_rate
0,Gross Domestic Product,2013,Q1,2013-Q1,712484.0,NaN
1,Gross Domestic Product,2013,Q2,2013-Q2,721647.0,NaN
2,Gross Domestic Product,2013,Q3,2013-Q3,729632.0,NaN
3,Gross Domestic Product,2013,Q4,2013-Q4,722821.0,NaN
4,Gross Domestic Product,2014,Q1,2014-Q1,762832.0,7.07
5,Gross Domestic Product,2014,Q2,2014-Q2,763677.0,5.82
6,Gross Domestic Product,2014,Q3,2014-Q3,759291.0,4.06
7,Gross Domestic Product,2014,Q4,2014-Q4,666024.0,-7.86


## Transformation 3 — Prepare Unemployment Data - Hanin

The unemployment dataset is already quarterly, so no aggregation is required. Only the required columns are selected for the next integration step.

In [11]:
# Select the unemployment columns needed for integration
unemployment_quarterly = unemployment[
    [
        "year",
        "quarter",
        "year_quarter",
        "unemployment_rate",
        "is_estimated"
    ]
].copy()

unemployment_quarterly.head()

,year,quarter,year_quarter,unemployment_rate,is_estimated
0,2013,Q1,2013-Q1,5.88,True
1,2013,Q2,2013-Q2,5.72,True
2,2013,Q3,2013-Q3,5.69,True
3,2013,Q4,2013-Q4,5.66,True
4,2014,Q1,2014-Q1,5.61,True


## Transformation Rules

| Rule ID | Description | Input Column(s) | Output Column |
|---|---|---|---|
| R1 | Aggregate monthly inflation data into quarterly average inflation | inflation_rate, year, quarter, year_quarter | quarterly_inflation_rate |
| R2 | Calculate year-over-year nominal GDP growth by comparing each quarter with the same quarter in the previous year | gdp_value | nominal_gdp_growth_rate |
| R3 | Prepare quarterly unemployment data without additional aggregation because the source is already quarterly | unemployment_rate, is_estimated, year, quarter, year_quarter | unemployment_quarterly |

In [12]:
# Test 1: Check that each quarter appears only once
assert inflation_quarterly["year_quarter"].is_unique

# Test 2: Check that quarterly inflation contains no missing values
assert inflation_quarterly["quarterly_inflation_rate"].notnull().all()

print("Inflation transformation tests passed.")

Inflation transformation tests passed.


In [13]:
# Test 1: Check that each quarter appears only once
assert gdp["year_quarter"].is_unique

# Test 2: Check that GDP values are not missing
assert gdp["gdp_value"].notnull().all()

# Test 3: Check that GDP growth is available after the first 4 quarters
assert gdp["nominal_gdp_growth_rate"].iloc[4:].notnull().all()

print("GDP transformation tests passed.")

GDP transformation tests passed.


In [14]:
# Test 1: Check that each quarter appears only once
assert unemployment_quarterly["year_quarter"].is_unique

# Test 2: Check that unemployment rates are not missing
assert unemployment_quarterly["unemployment_rate"].notnull().all()

# Test 3: Check that unemployment rates are within the valid range
assert unemployment_quarterly["unemployment_rate"].between(0, 100).all()

print("Unemployment transformation tests passed.")

Unemployment transformation tests passed.


In [15]:
# Test empty input

empty_data = pd.DataFrame(
    columns=["year", "quarter", "year_quarter"]
)

assert empty_data.empty

print("Empty input test passed.")

Empty input test passed.


In [16]:
# Test null join keys

assert gdp["year_quarter"].notnull().all()
assert inflation_quarterly["year_quarter"].notnull().all()
assert unemployment_quarterly["year_quarter"].notnull().all()

print("Null key tests passed.")

Null key tests passed.


In [17]:
# Test valid quarter values

valid_quarters = ["Q1", "Q2", "Q3", "Q4"]

assert gdp["quarter"].isin(valid_quarters).all()
assert inflation_quarterly["quarter"].isin(valid_quarters).all()
assert unemployment_quarterly["quarter"].isin(valid_quarters).all()

print("Quarter edge case test passed.")

Quarter edge case test passed.
